In [12]:
from hana_ml import dataframe
url, port, user, pwd = "810070ba-df1a-4553-9a82-23690dc0158e.hana.demo-hc-3-haas-hc-dev.dev-aws.hanacloud.ondemand.com", \
443, "SAPUSER", "nIL0yr8S"
cc = dataframe.ConnectionContext(url, port, user, pwd)

In [13]:
import numpy as np
import pandas as pd
x1 = np.arange(1, 21) / 20
x2 = np.sqrt(x1)
query_data = pd.DataFrame(dict(TS_ID=[0] * 10 + [1] * 10, TS_ORDER=range(20), TS_VAL=x2))
ref_data = pd.DataFrame(dict(TS_ID=['A'] * 10 + ['B'] * 10, TS_ORDER=range(20), TS_VAL=x1))

In [14]:
from hana_ml.dataframe import create_dataframe_from_pandas
query_df = create_dataframe_from_pandas(cc, query_data,
                                        "DTW_QUERY_SIM_DATA_TBL",
                                        force=True)
ref_df = create_dataframe_from_pandas(cc, ref_data,
                                      "DTW_REF_SIM_DATA_TBL",
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.73it/s]


In [15]:
from hana_ai.tools.hana_ml_tools.dtw_tools import DTW
dtw_tool = DTW(cc)

In [16]:
dtw_input = dict(query_table="DTW_QUERY_SIM_DATA_TBL",
                 query_ts_id='TS_ID',
                 query_ts_order='TS_ORDER',
                 ref_table="DTW_REF_SIM_DATA_TBL",
                 ref_ts_id='TS_ID',
                 ref_ts_order='TS_ORDER',
                 radius=3, save_alignment=True,
                 step_pattern=2)
dtw_tool.run(tool_input=dtw_input)

'{"DTW_results_in_tuple(QUERY_TS_ID, REF_TS_ID, DISTANCE, WEIGHT, AVG_DISTANCE)": "[(0, \'A\', 1.2240597361728045, 10.0, 0.12240597361728045), (1, \'A\', 4.964872092360034, 10.0, 0.4964872092360034), (0, \'B\', 1.5259402638271957, 10.0, 0.15259402638271957), (1, \'B\', 0.30533106119102316, 10.0, 0.030533106119102316)]", "dtw_alignment_table": "DTW_QUERY_SIM_DATA_TBL_DTW_REF_SIM_DATA_TBL_DTW_ALIGNMENT"}'

In [17]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [dtw_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

In [18]:
instruction = "I want to compute the dynamic time warping (DTW) distances between times series, "+\
"where the query table is DTW_QUERY_SIM_DATA_TBL, " +\
"the reference table is DTW_REF_SIM_DATA_TBL, query_ts_id is TS_ID, query_ts_order is TS_ORDER," +\
"ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3, step pattern is 5 and the alignments between time-series should be saved."
agent_chain.invoke(instruction)

{'input': 'I want to compute the dynamic time warping (DTW) distances between times series, where the query table is DTW_QUERY_SIM_DATA_TBL, the reference table is DTW_REF_SIM_DATA_TBL, query_ts_id is TS_ID, query_ts_order is TS_ORDER,ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3, step pattern is 5 and the alignments between time-series should be saved.',
 'output': "The dynamic time warping (DTW) distances between the time series have been computed. The results are as follows: [(0, 'A', 1.5990597361728045, 10.0, 0.15990597361728046), (1, 'A', 5.339872092360034, 10.0, 0.5339872092360034), (0, 'B', 2.0509402638271954, 10.0, 0.20509402638271954), (1, 'B', 0.5286511142591743, 10.0, 0.05286511142591743)]. The alignment information is saved in the table 'DTW_QUERY_SIM_DATA_TBL_DTW_REF_SIM_DATA_TBL_DTW_ALIGNMENT'."}

In [ ]:
instruction = "I want to compute the dynamic time warping (DTW) distances between times series, "+\
"where the query table is DTW_QUERY_SIM_DATA_TBL, " +\
"the reference table is DTW_REF_SIM_DATA_TBL, query_ts_id is TS_ID, query_ts_order is TS_ORDER, " +\
"ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3, aligment method is 'closed'"
agent_chain.invoke(instruction)

{'input': "I want to compute the dynamic time warping (DTW) distances between times series, where the query table is DTW_QUERY_SIM_DATA_TBL, the reference table is DTW_REF_SIM_DATA_TBL, query_ts_id is TS_ID, query_ts_order is TS_ORDER,ref_ts_id is TS_ID, ref_ts_order is TS_ORDER, radius is 3, aligment method is 'closed'",
 'output': "The dynamic time warping (DTW) distances between the time series have been computed. Here are the results: \n\n- For query time series 0 and reference time series 'A', the DTW distance is approximately 1.98, with an average distance of approximately 0.10. \n- For query time series 1 and reference time series 'A', the DTW distance is approximately 9.59, with an average distance of approximately 0.50. \n- For query time series 0 and reference time series 'B', the DTW distance is approximately 2.95, with an average distance of approximately 0.16. \n- For query time series 1 and reference time series 'B', the DTW distance is approximately 0.63, with an average

In [20]:
cc.drop_table('DTW_QUERY_SIM_DATA_TBL_DTW_REF_SIM_DATA_TBL_DTW_ALIGNMENT')
cc.drop_table('DTW_QUERY_SIM_DATA_TBL')
cc.drop_table('DTW_REF_SIM_DATA_TBL')

In [21]:
cc.close()